[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/edit-state-human-feedback.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239520-lesson-3-editing-state-and-human-feedback)

# 编辑图状态

## 回顾

我们讨论了人机协同（human-in-the-loop）的动机：

(1) `Approval`（审批）—— 我们可以中断 Agent，将状态展示给用户，并允许用户批准某个操作

(2) `Debugging`（调试）—— 我们可以回退图的执行以重现或避免问题

(3) `Editing`（编辑）—— 你可以修改状态

我们展示了断点如何支持用户审批，但还不知道如何在图被中断后修改图状态！

## 目标

现在，让我们展示如何直接编辑图状态并插入人工反馈。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

## 编辑状态

之前，我们介绍了断点。

我们使用断点来中断图，并在执行下一个节点之前等待用户审批。

但断点也是[修改图状态的机会](https://docs.langchain.com/oss/javascript/langgraph/use-time-travel#3-update-the-state)。

让我们在 `assistant` 节点之前设置一个断点来配置我们的 Agent。

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """计算 a 和 b 的乘积。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将作为一个工具
def add(a: int, b: int) -> int:
    """计算 a 和 b 的和。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """计算 a 除以 b 的商。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatDeepSeek(model="deepseek-chat")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些边决定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果来自 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["assistant"], checkpointer=memory)

# 展示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

让我们运行！

我们可以看到图在聊天模型响应之前被中断了。

In [ ]:
# 输入
initial_input = {"messages": "Multiply 2 and 3"}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
state = graph.get_state(thread)
state

现在，我们可以直接应用状态更新。

请记住，对 `messages` 键的更新将使用 `add_messages` reducer：

* 如果我们想要覆盖现有消息，可以提供消息 `id`。
* 如果我们只想追加到消息列表中，则可以传入未指定 `id` 的消息，如下所示。

In [ ]:
graph.update_state(
    thread,
    {"messages": [HumanMessage(content="No, actually multiply 3 and 3!")]},
)

让我们看一下。

我们使用一条新消息调用了 `update_state`。

`add_messages` reducer 将其追加到我们的状态键 `messages` 中。

In [ ]:
new_state = graph.get_state(thread).values
for m in new_state['messages']:
    m.pretty_print()

现在，让我们继续运行 Agent，只需传入 `None` 让其从当前状态继续。

我们发出当前状态，然后继续执行剩余的节点。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们回到了 `assistant` 节点，该节点设有我们的 `breakpoint`。

我们可以再次传入 `None` 来继续。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

### 在 Studio 中编辑图状态

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

LangGraph API [支持编辑图状态](https://docs.langchain.com/langsmith/add-human-in-the-loop)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
# 这是本地开发服务器的 URL
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

我们的 Agent 定义在 `studio/agent.py` 中。

如果你查看代码，会发现它*没有*断点！

当然，我们可以将断点添加到 `agent.py` 中，但 API 的一个非常好的功能是我们可以传入断点！

这里，我们传入 `interrupt_before=["assistant"]`。

In [ ]:
initial_input = {"messages": "Multiply 2 and 3"}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    "agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"Receiving new event of type: {chunk.event}...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

我们可以获取当前状态

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state

我们可以查看状态中的最后一条消息。

In [ ]:
last_message = current_state['values']['messages'][-1]
last_message

我们可以编辑它！

In [ ]:
last_message['content'] = "No, actually multiply 3 and 3!"
last_message

In [ ]:
last_message

请记住，正如我们之前所说的，对 `messages` 键的更新将使用相同的 `add_messages` reducer。

如果我们想要覆盖现有消息，则可以提供消息 `id`。

这里，我们做到了这一点。如上所示，我们只修改了消息 `content`。

In [ ]:
await client.threads.update_state(thread['thread_id'], {"messages": last_message})

现在，我们通过传入 `None` 来恢复执行。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"Receiving new event of type: {chunk.event}...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

我们得到工具调用的结果为 `9`，正如预期。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"Receiving new event of type: {chunk.event}...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

## 等待用户输入

很明显，我们可以在断点之后编辑 Agent 状态。

现在，如果我们希望允许人工反馈来执行此状态更新，该怎么办？

我们将添加一个节点，作为 Agent 中人工反馈的占位符。

这个 `human_feedback` 节点允许用户直接将反馈添加到状态。

我们使用 `interrupt_before` 在 `human_feedback` 节点之前设置断点。

我们设置一个 checkpointer 来保存图在此节点之前的状态。

In [ ]:
# 系统消息
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 空操作节点，在此处中断
def human_feedback(state: MessagesState):
    pass

# Assistant 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_node("human_feedback", human_feedback)

# 定义边：这些边决定控制流
builder.add_edge(START, "human_feedback")
builder.add_edge("human_feedback", "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果来自 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "human_feedback")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["human_feedback"], checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

我们将从用户那里获取反馈。

我们像之前一样使用 `.update_state` 来用获取到的人类反馈更新图的状态。

我们使用 `as_node="human_feedback"` 参数将此状态更新应用为指定节点 `human_feedback` 的输出。

In [ ]:
# 输入
initial_input = {"messages": "Multiply 2 and 3"}

# 线程
thread = {"configurable": {"thread_id": "5"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()
    
# 获取用户输入
user_input = input("请告诉我你想如何更新状态: ")

# 我们现在以 human_feedback 节点的身份更新状态
graph.update_state(thread, {"messages": user_input}, as_node="human_feedback")

# 继续图的执行
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

In [ ]:
# 继续图的执行
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()